####**로지스틱회귀분석 결과 해석**

**로그-우도**
- 모형의 설명력으로 모델의 성능평가에 사용함
- 로그-우도 : model.llf
- [250514수정] **로그우도가 0에 가까울수록 높은 적합도, 더 작은 음수일수록 낮은 적합도임**

**잔차이탈도(Residual Deviance)**
- 모형이 데이터를 얼마나 잘 설명하는지 평가하는 지표로 0이상의 양수 값을 갖음
- model.deviance, $-2 * (LL_{\text{fitted}} - LL_{\text{saturated}})$
- $LL_{\text{fitted}}$: 현재 적합된(fitted) 모델의 로그 우도
- $LL_{\text{saturated}}$: 포화 모델(saturated model)의 로그 우도
  - 잔차가 없고 최대 우도를 가지는 모델의 로그 우도로 모든 관측값을 완벽하게 설명하는 모델
- 잔차이탈도가 낮을수록 모델이 데이터를 잘 설명
- 잔차이탈도 / 자유도 = 1에 가까우면 적절한 것이고 1보다 크면 과소적합, 1보다 작으면 과대적합 가능성이 있음

**오즈(Odds)**
- 독립변수의 변화에 따라 종속변수가 발생할 확률과 발생하지 않을 확률의 비율(성공과 실패의 확률 비율)
- Odds = P(Y=1) / P(Y=0) = 성공확률 / 실패 확률
- 오즈는 확률의 비율로 항상 0 이상의 값을 갖는다.

**오즈비(Odds Ratio)**
- 특정 독립변수가 1 단위 증가할 때, 종속변수가 발생할 오즈(odds)가 몇 배 증가(또는 감소)하는지를 나타냅니다.
- 특정 변수의 오즈비 : np.exp(model.params['변수명'])
- 특정 변수가 n 증가시 성공의 오즈는 몇 배 증가하는가? : np.exp(model.params['변수명']*5)
- 오즈비가 1보다 크면 Y=1의 가능성이 증가를 의미한다.
- 즉, 독립변수가 증가할수록 종속변수가 발생할 확률이 높아진다.
- 오즈비가 1이면 독립변수는 종속변수에 영향을 미치지 않음
- 오즈비가 1보다 작으면 독립변수가 증가할수록 종속변수 Y=1의 가능성이 감소함(독립변수가 증가할수록 종속변수가 발생할 확률이 낮아진다.)

**유의확률(p-value)**
- 독립변수의 통계적 유의미 판단에 사용함
- p-value가 작을수록 통계적 유의미성은 강한 것임
- 종속변수의 로그 오즈에 통계적으로 유의미한 영향을 미친다.
  - 귀무가설 : 회귀 모델의 기울기가 0이다 즉, 독립변수가 종속변수에 영향을 미치지 않는다.
  - 대립가설 : 회귀 모델의 기울기가 0이 아니다. 즉, 독립변수가 종속변수에 영향을 미친다.
  - 귀무가설 기각 조건: model.pvalues['변수명'] <= 유의수준(0.05)


### formula 만들기
- formula=`'종속변수 ~ 독립변수1 + 독립변수2 + 독립변수3 ...'`
- 상수항 기본으로 포함
 - 상수항을 불포함하기 위해 -1 사용
- 범주형 변수에 대해 더미변수로 만들기
 - `C(독립변수)`
 - 범주명 기준 오름차순 정렬시 첫 번째 범주 제외 나머지 범주만 더미변수로 사용됨
- 교호작용(Interaction)항 사용
 - `독립변수1 * 독립변수2` : 개별 효과(독립변수1, 독립변수2) + 교호작용 포함
 - `독립변수1:독립변수2` : 순수한 교호작용만 포함
- 다항식(Polynomial) 포함
 - `I(독립변수 **2)`

### Day2. 로지스틱 회귀 (유방암 데이터셋)
유방암(breast_cancer) 데이터를 사용해 로지스틱 회귀를 수행합니다.

In [1]:
import pandas as pd
pd.set_option('display.width', 120)

path = "https://raw.githubusercontent.com/Soyoung-Yoon/data_01/main/"


In [2]:
df = pd.read_csv(path + "breast_cancer04.csv")
print(df.head(3))

   mean_radius  mean_texture  mean_area  mean_smoothness  mean_concave_points  radius_error  area_error  \
0       24.630         21.60     1841.0          0.10300              0.14710        0.9915      139.90   
1        8.888         14.64      244.0          0.09783              0.02872        0.5262       25.44   
2        9.847         15.68      293.2          0.09492              0.02416        0.2498       15.24   

   compactness_error  worst_radius  worst_texture  worst_concavity  worst_fractal_dimension  target  
0            0.03212        29.920          26.93          0.46580                  0.09671       0  
1            0.09368         9.733          15.67          0.14340                  0.10840       1  
2            0.02042        11.240          22.99          0.08434                  0.09209       1  


In [3]:
# 2-1) 종속변수는 'target' 입니다.
# 범주의 종류 및 개수를 확인해 봅니다.
print(df['target'].value_counts())

target
1    357
0    212
Name: count, dtype: int64


In [ ]:
# 2-2) 순서대로 398개는 train 데이터, 171개는 test 데이터로 사용합니다.
# train, test 분리 후, shape을 출력해서 확인하세요.
train = df.iloc[:398, :]
test = df.iloc[399:, :]
print(train.shape, test.shape) # (398, 13) (170, 13)

(398, 13) (170, 13)


다음과 같은 로지스틱회귀 모형을 사용한 분류모델을 만들고 결과를 확인합니다.
- breast_cancer04.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 하며, 규제는 사용하지 않습니다.
- 종속변수 : target
- 독립변수 : target을 제외한 모든 나머지 변수

In [13]:
# 2-3) GLM.from_formula() 를 사용해 분석하려고 합니다.
# formula를 작성하고, train 데이터를 사용해 로지스틱 회귀모형을 생성합니다.
# formula를 만들때 str.join()을 사용해 만들면 편하게 만들 수 있습니다.
# 생성 후, model.summary()를 출력해 봅니다.
from statsmodels.api import GLM, add_constant, families
# print(df.head(3))
formula = 'target ~ ' + ' + '.join(df.loc[:, 'mean_radius':'worst_fractal_dimension'])
# print(formula)
model = GLM.from_formula(formula, df, family=families.Binomial()).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                  569
Model:                            GLM   Df Residuals:                      556
Model Family:                Binomial   Df Model:                           12
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -29.940
Date:                Wed, 05 Nov 2025   Deviance:                       59.880
Time:                        18:44:02   Pearson chi2:                     136.
No. Iterations:                    11   Pseudo R-squ. (CS):             0.7034
Covariance Type:            nonrobust                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

In [52]:
# 2-4) model에서 유의미한 설명변수만 선택하고, 그 개수를 출력합니다.
# 유의수준 0.05을 사용한다.
print(round(model.pvalues[1:],3))
print(sum(round(model.pvalues[1:],2) <= 0.05)) # 4

mean_radius                0.053
mean_texture               0.190
mean_area                  0.333
mean_smoothness            0.384
mean_concave_points        0.005
radius_error               0.342
area_error                 0.074
compactness_error          0.001
worst_radius               0.003
worst_texture              0.129
worst_concavity            0.002
worst_fractal_dimension    0.706
dtype: float64
5


In [35]:
# 2-5) 학습데이터(train)를 사용하여, 2-4)에서 찾은 유의미한 설명변수만으로
# 로지스틱 회귀모형을 만듭니다.
# 생성 후, model.summary()를 출력해 봅니다.
train2 = train[['mean_concave_points','compactness_error','worst_radius','worst_concavity','target']]
formula2 = "target ~ " + ' + '.join(train2.columns)
model2 = GLM.from_formula(formula2, train2, family=families.Binomial()).fit()
print(model2.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                  398
Model:                            GLM   Df Residuals:                      392
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1541e-09
Date:                Wed, 05 Nov 2025   Deviance:                   2.3090e-09
Time:                        18:56:18   Pearson chi2:                 1.15e-09
No. Iterations:                    25   Pseudo R-squ. (CS):             0.7328
Covariance Type:            nonrobust                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept             -26.5661   1

/Users/rhkmbp/miniconda3/envs/study/lib/python3.10/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)


In [ ]:
# 2-6) 설명변수의 가장 높은 p-value를 구하여 주세요.
# 반올림하여 소수점 아래 3자리까지 출력합니다.
print(model2.pvalues[1:].max()) # 1.0

1.0


In [40]:
# 2-7) 유의수준 0.05하에서 유의성이 낮은 변수의 개수는 몇 개인가요?
print(sum(model2.pvalues[1: ] <= 0.05)) # 0

0


In [47]:
# 2-8) 평가 데이터를 사용하여 정확도를 구해 반올림하여 소수점 아래 3자리까지 출력합니다.
from sklearn.metrics import accuracy_score
# print(test['target'])
y_true = train2['target']
y_pred = model2.predict(train2).round().astype('int32')
acc = accuracy_score(y_true, y_pred)
print(round(acc, 3)) # 1.000

1.0


In [55]:
# 2-9) 아래의 sample을 사용하여 P(Y=0)에 대한 확률을 구하고,
# 반올림하여 소수점 아래 3자리까지 출력합니다.
# sample => mean_radius : 14.2, mean_area: 650.0, mean_concave_points : 0.05,
# compactness_error : 0.02, worst_radius : 16.5
sample = pd.DataFrame({'const': [0],
                       'mean_radius': [14.2],
                       'mean_area': [650.0],
                       'mean_concave_points': [0.05], 
                       'compactness_error': [0.02],
                       'worst_radius': [16.5]})
# print(sample)

p_y1 = model2.predict(sample)[0]
p_y0 = 1 - p_y1
print(round(p_y0, 3))

PatsyError: predict requires that you use a DataFrame when predicting from a model
that was created using the formula api.

The original error message returned by patsy is:
Error evaluating factor: NameError: name 'worst_concavity' is not defined
    target ~ mean_concave_points + compactness_error + worst_radius + worst_concavity + target
                                                                      ^^^^^^^^^^^^^^^

In [ ]:
#2-10) 위의 sample에 대한 오즈(odds)는?
# 결과는 반올림하여 소수점 아래 3자리까지 출력합니다.

In [56]:
#2-11) 'mean_area'을 설명변수로 하였을 때의 오즈비(Odds Ratio)는?
# 결과는 반올림하여 소수점 아래 3자리까지 출력합니다.
import numpy as np
res = np.exp(model2.params['mean_area'])
print(round(res, 3))

KeyError: 'mean_area'

In [57]:
#2-12) 'mean_radius'가 2증가하면 오즈는 몇 배 증가하는가?
# 반올림하여 정수로 출력합니다.
res = np.exp(model2.params['mean_radius'] * 2)
print(round(res))


KeyError: 'mean_radius'

In [ ]:
#2-13) test 데이터에 대한 roc_auc 점수를 구합니다.
# 결과는 반올림하여 소수점 아래 3자리까지 출력합니다.
